# 🧠 AlgorimSeek — Text Model Training
**Model:** DeepSeek-R1 / Qwen2.5-Coder (7B / 3B / 1.5B / 0.5B)  
**Dataset:** `algorim_training.json` (1 034 chat records) + `1000_code_dataset` (.algo files)  
**Method:** QLoRA fine-tuning via Unsloth + TRL SFTTrainer  
**Output:** LoRA adapter → Merged HF model → GGUF F16 → GGUF Q4_K_M


## ⚙️ 1 · GPU Check

In [ ]:
import subprocess, torch
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


## 📦 2 · Install Dependencies

In [ ]:
%%capture
# Unsloth — memory-efficient fine-tuning engine
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes xformers
!pip install datasets transformers sentencepiece protobuf huggingface_hub
print("✅ All packages installed")


## 📁 3 · Upload & Extract Datasets

In [ ]:
import os, zipfile, json, glob
from google.colab import files

# ── Option A: upload manually ──────────────────────────────────────
print("Upload your dataset ZIPs (1000_code_dataset.zip + Datasets.zip):")
uploaded = files.upload()

# ── Option B: mount Google Drive ──────────────────────────────────
# from google.colab import drive
# drive.mount('/content/drive')
# DRIVE_PATH = '/content/drive/MyDrive/AlgorimDatasets'
# !cp {DRIVE_PATH}/1000_code_dataset.zip /content/
# !cp {DRIVE_PATH}/Datasets.zip /content/

# Extract
os.makedirs('/content/datasets', exist_ok=True)
for z in ['1000_code_dataset.zip', 'Datasets.zip']:
    if os.path.exists(f'/content/{z}'):
        with zipfile.ZipFile(f'/content/{z}') as zf:
            zf.extractall('/content/datasets')
        print(f"✅ Extracted {z}")

# Verify
algo_files = glob.glob('/content/datasets/**/*.algo', recursive=True)
json_files = glob.glob('/content/datasets/**/*.json', recursive=True)
print(f"\n📂 .algo files: {len(algo_files)}")
print(f"📂 .json files: {len(json_files)}")
for j in json_files:
    print(f"   {j}")


## 🧩 4 · Dataset Processing

In [ ]:
import json, glob, random
from datasets import Dataset

# ── Load chat records (algorim_training.json) ─────────────────────
chat_records = []
training_json = glob.glob('/content/datasets/**/*training.json', recursive=True)
if not training_json:
    training_json = glob.glob('/content/datasets/**/*.json', recursive=True)
    training_json = [f for f in training_json if 'training' in f and 'vision' not in f]

if training_json:
    with open(training_json[0]) as f:
        for line in f:
            line = line.strip()
            if line:
                try:
                    chat_records.append(json.loads(line))
                except:
                    pass
    print(f"✅ Loaded {len(chat_records)} chat records")
else:
    print("⚠️  algorim_training.json not found — check your upload")

# ── Load .algo files as continuation pretraining ──────────────────
algo_files = sorted(glob.glob('/content/datasets/**/*.algo', recursive=True))
algo_texts = []
for f in algo_files:
    with open(f, encoding='utf-8', errors='ignore') as fh:
        txt = fh.read().strip()
    if txt:
        algo_texts.append(txt)
print(f"✅ Loaded {len(algo_texts)} .algo source files")

# ── Build unified SFT samples ─────────────────────────────────────
SYSTEM_PROMPT = (
    "You are AlgorimSeek, an AI expert in the Algorim programming language (algo a47). "
    "You understand algorithmic pseudocode, generate code images, compile to bytecode, "
    "and support /imagine /compile /debug /execute commands."
)

def format_chat_sample(record):
    msgs = record.get('messages', [])
    parts = []
    for m in msgs:
        role, content = m['role'], m['content']
        if role == 'system':
            parts.append(f"<|im_start|>system\n{content}<|im_end|>")
        elif role == 'user':
            parts.append(f"<|im_start|>user\n{content}<|im_end|>")
        elif role == 'assistant':
            parts.append(f"<|im_start|>assistant\n{content}<|im_end|>")
    return "\n".join(parts)

def format_algo_as_chat(algo_code):
    # Wrap .algo file as a code-generation sample
    lines = algo_code.strip().split('\n')
    action_name = "the action" 
    for line in lines[:3]:
        if line.strip().startswith('Action '):
            action_name = line.strip().split('(')[0].replace('Action ','')
            break
    user_msg = f"Show me the Algorim implementation of {action_name}."
    return (
        f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n"
        f"<|im_start|>user\n{user_msg}<|im_end|>\n"
        f"<|im_start|>assistant\n{algo_code}<|im_end|>"
    )

all_samples = []
for rec in chat_records:
    txt = format_chat_sample(rec)
    if txt:
        all_samples.append({"text": txt})

for algo in algo_texts:
    txt = format_algo_as_chat(algo)
    all_samples.append({"text": txt})

random.shuffle(all_samples)
print(f"\n📊 Total training samples: {len(all_samples)}")
print(f"   Chat records: {len(chat_records)}")
print(f"   Algo files:   {len(algo_texts)}")
print("\nSample preview:")
print(all_samples[0]['text'][:400])


## 🤖 5 · Model Selection

In [ ]:
# ── Choose your model ─────────────────────────────────────────────
# Uncomment ONE line to select the base model

MODEL_CONFIGS = {
    # DeepSeek-R1 Distill series (reasoning + code)
    "DeepSeek-R1-7B":   "unsloth/DeepSeek-R1-Distill-Qwen-7B-unsloth-bnb-4bit",
    "DeepSeek-R1-1.5B": "unsloth/DeepSeek-R1-Distill-Qwen-1.5B-bnb-4bit",
    # Qwen2.5-Coder series (code-focused)
    "Qwen2.5-Coder-7B": "unsloth/Qwen2.5-Coder-7B-Instruct-bnb-4bit",
    "Qwen2.5-Coder-3B": "unsloth/Qwen2.5-Coder-3B-Instruct-bnb-4bit",
    "Qwen2.5-Coder-0.5B":"unsloth/Qwen2.5-Coder-0.5B-Instruct-bnb-4bit",
    # Qwen2.5 general
    "Qwen2.5-7B":       "unsloth/Qwen2.5-7B-Instruct-bnb-4bit",
    "Qwen2.5-3B":       "unsloth/Qwen2.5-3B-Instruct-bnb-4bit",
    "Qwen2.5-0.5B":     "unsloth/Qwen2.5-0.5B-Instruct-bnb-4bit",
}

SELECTED = "DeepSeek-R1-7B"   # ← change this line

BASE_MODEL  = MODEL_CONFIGS[SELECTED]
OUTPUT_NAME = "AlgorimSeek"
MAX_SEQ_LEN = 2048

# LoRA config
LORA_R     = 16
LORA_ALPHA = 32

print(f"✅ Selected: {SELECTED}")
print(f"   HF model: {BASE_MODEL}")
print(f"   Output:   {OUTPUT_NAME}")


## 🔧 6 · Load Model + Apply LoRA

In [ ]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name    = BASE_MODEL,
    max_seq_length= MAX_SEQ_LEN,
    dtype         = None,          # auto-detect
    load_in_4bit  = True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r              = LORA_R,
    target_modules = ["q_proj","k_proj","v_proj","o_proj",
                       "gate_proj","up_proj","down_proj"],
    lora_alpha     = LORA_ALPHA,
    lora_dropout   = 0,
    bias           = "none",
    use_gradient_checkpointing = "unsloth",
    random_state   = 42,
)

model.print_trainable_parameters()
print(f"\n✅ Model loaded — {SELECTED}")


## 📋 7 · Prepare HF Dataset

In [ ]:
from datasets import Dataset

hf_dataset = Dataset.from_list(all_samples)

# Train / validation split
split = hf_dataset.train_test_split(test_size=0.05, seed=42)
train_ds = split['train']
eval_ds  = split['test']

print(f"Train: {len(train_ds):,} samples")
print(f"Eval:  {len(eval_ds):,} samples")

# Verify tokenisation
sample_enc = tokenizer(train_ds[0]['text'], return_tensors='pt')
print(f"\nSample token length: {sample_enc['input_ids'].shape[1]}")


## 🏋️ 8 · Training

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_ds,
    eval_dataset  = eval_ds,
    args = SFTConfig(
        dataset_text_field = "text",
        max_seq_length     = MAX_SEQ_LEN,
        per_device_train_batch_size  = 2,
        gradient_accumulation_steps  = 4,
        warmup_steps         = 10,
        num_train_epochs     = 3,
        learning_rate        = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 =     torch.cuda.is_bf16_supported(),
        logging_steps        = 20,
        evaluation_strategy  = "steps",
        eval_steps           = 100,
        save_strategy        = "steps",
        save_steps           = 200,
        output_dir           = f"/content/{OUTPUT_NAME}-checkpoints",
        optim                = "adamw_8bit",
        weight_decay         = 0.01,
        lr_scheduler_type    = "cosine",
        seed                 = 42,
        report_to            = "none",
    ),
)

print("🚀 Starting training...")
trainer_stats = trainer.train()
print(f"\n✅ Training complete!")
print(f"   Runtime: {trainer_stats.metrics['train_runtime']:.0f}s")
print(f"   Train loss: {trainer_stats.metrics['train_loss']:.4f}")


## 💾 9 · Save LoRA Adapter

In [ ]:
lora_path = f"/content/{OUTPUT_NAME}-lora"
model.save_pretrained(lora_path)
tokenizer.save_pretrained(lora_path)
print(f"✅ LoRA adapter saved → {lora_path}")
import os
total = sum(os.path.getsize(os.path.join(r,f)) for r,d,fs in os.walk(lora_path) for f in fs)
print(f"   Size: {total/1e6:.1f} MB")


## 🔀 10 · Merge + Save Full Model (FP16)

In [ ]:
merged_path = f"/content/{OUTPUT_NAME}"
print("⏳ Merging LoRA weights into base model...")
model.save_pretrained_merged(merged_path, tokenizer, save_method="merged_16bit")
print(f"\n✅ Merged model saved → {merged_path}")
import os
total = sum(os.path.getsize(os.path.join(r,f)) for r,d,fs in os.walk(merged_path) for f in fs)
print(f"   Size: {total/1e9:.2f} GB")


## 📤 11 · Export GGUF — F16

In [ ]:
gguf_path = f"/content/{OUTPUT_NAME}-GGUF"
print("⏳ Converting to GGUF F16...")
model.save_pretrained_gguf(gguf_path, tokenizer, quantization_method="f16")
import glob
f16_files = glob.glob(f"{gguf_path}/*.gguf")
print(f"\n✅ GGUF F16 saved:")
for f in f16_files:
    size_gb = os.path.getsize(f)/1e9
    print(f"   {f}  ({size_gb:.2f} GB)")


## 📤 12 · Export GGUF — Q4_K_M

In [ ]:
print("⏳ Quantising to GGUF Q4_K_M...")
model.save_pretrained_gguf(gguf_path, tokenizer, quantization_method="q4_k_m")
import glob
q4_files = [f for f in glob.glob(f"{gguf_path}/*.gguf") if 'Q4' in f.upper() or 'q4' in f]
print(f"\n✅ GGUF Q4_K_M saved:")
for f in q4_files:
    size_gb = os.path.getsize(f)/1e9
    print(f"   {f}  ({size_gb:.2f} GB)")
print("\n🎉 AlgorimSeek training pipeline complete!")
print(f"  LoRA:      {lora_path}")
print(f"  Merged:    {merged_path}")
print(f"  GGUF dir:  {gguf_path}")


## ☁️ 13 · (Optional) Push to Hugging Face

In [ ]:
# Uncomment and fill in your HF token + repo name
# from huggingface_hub import login
# login(token="hf_YOUR_TOKEN_HERE")
# HF_REPO = "YourHFUsername/AlgorimSeek-7B"
# model.push_to_hub(HF_REPO)
# tokenizer.push_to_hub(HF_REPO)
# model.push_to_hub_gguf(HF_REPO, tokenizer, quantization_method=["f16","q4_k_m"])
print("HF push disabled — uncomment the block above to enable.")


## 🧪 14 · Quick Inference Test

In [ ]:
from unsloth import FastLanguageModel
FastLanguageModel.for_inference(model)

TEST_PROMPT = (
    "<|im_start|>system\n"
    "You are AlgorimSeek, an AI expert in the Algorim programming language (algo a47). "
    "You understand algorithmic pseudocode, generate code images, compile to bytecode, "
    "and support /imagine /compile /debug /execute commands.<|im_end|>\n"
    "<|im_start|>user\n"
    "Write an Algorim action to compute the factorial of n recursively.<|im_end|>\n"
    "<|im_start|>assistant\n"
)

inputs = tokenizer(TEST_PROMPT, return_tensors="pt").to("cuda")
outputs = model.generate(
    **inputs,
    max_new_tokens = 300,
    temperature    = 0.7,
    do_sample      = True,
    pad_token_id   = tokenizer.eos_token_id,
)
response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
print("=" * 60)
print("AlgorimSeek response:")
print("=" * 60)
print(response)
